# geoconformal Tutorial: GeoCP & GeoSIMCP

This notebook demonstrates how to use the `geoconformal` package for **model-agnostic uncertainty quantification** in spatial prediction, using the Seattle housing price dataset as an example.

We will cover:
1. **Data loading & preparation** — how to split data into train / calibration / test
2. **GeoCP** — uncertainty quantification using geographic distance only
3. **GeoSIMCP** — uncertainty quantification using geographic distance + feature similarity
4. **Hyperparameter tuning** — grid search for optimal bandwidth and lambda
5. **Spatial mapping** — visualizing prediction and uncertainty on a map

## 0. Install & Import

In [ ]:
# Uncomment to install (e.g. on Google Colab)
# !pip install geoconformal xgboost contextily matplotlib

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

from geoconformal import GeoConformalSpatialRegression, GeoSIMConformalSpatialRegression

---
## 1. Data Loading & Preparation

The Seattle housing dataset contains 3,000 property records with:
- **Features**: bathrooms, sqft_living, sqft_lot, grade, condition, waterfront, view, age
- **Coordinates**: UTM_X, UTM_Y (EPSG:32610)
- **Target**: log_price (log10 of sale price)

In [ ]:
# ---------- Load data ----------
data = pd.read_csv('seattle_sample_3k.csv', index_col=0)
print(f"Dataset shape: {data.shape}")
data.head()

In [ ]:
# ---------- Define variables ----------
feature_cols = ['bathrooms', 'sqft_living', 'sqft_lot', 'grade',
                'condition', 'waterfront', 'view', 'age']
coord_cols   = ['UTM_X', 'UTM_Y']
target_col   = 'log_price'

X      = data[feature_cols].values           # features (for prediction)
coords = data[coord_cols].values              # geographic coordinates
y      = data[target_col].values              # target variable

### Data splitting: Train / Calibration / Validation / Test

Conformal prediction requires a **calibration set** that is separate from the training set.
Additionally, a **validation set** is needed for hyperparameter tuning (bandwidth, lambda) to avoid data leakage on the test set.

```
Full Dataset (3000 samples)
  ├── Training set     (70% = 2100)  → train the prediction model
  ├── Calibration set  (10% =  300)  → compute nonconformity scores for uncertainty
  ├── Validation set   (10% =  300)  → tune hyperparameters (bandwidth, lambda)
  └── Test set         (10% =  300)  → final evaluation of uncertainty estimates
```

**Important**: Hyperparameters should be tuned on the validation set, NOT the test set.

In [ ]:
# ---------- Split: 70% train, 10% calibration, 10% validation, 10% test ----------
X_train, X_temp, y_train, y_temp, coords_train, coords_temp = \
    train_test_split(X, y, coords, test_size=0.3, random_state=42)

X_calib, X_temp2, y_calib, y_temp2, coords_calib, coords_temp2 = \
    train_test_split(X_temp, y_temp, coords_temp, test_size=2/3, random_state=42)

X_val, X_test, y_val, y_test, coords_val, coords_test = \
    train_test_split(X_temp2, y_temp2, coords_temp2, test_size=0.5, random_state=42)

print(f"Train: {X_train.shape[0]},  Calib: {X_calib.shape[0]},  Val: {X_val.shape[0]},  Test: {X_test.shape[0]}")

---
## 2. Train a Prediction Model

We train an XGBoost model **using features + coordinates** as inputs. 
`geoconformal` is model-agnostic — you can replace this with Random Forest, SVR, MLP, or any model.

In [ ]:
# ---------- Combine features and coordinates for spatial prediction ----------
X_train_spatial = np.hstack([X_train, coords_train])
X_calib_spatial = np.hstack([X_calib, coords_calib])
X_val_spatial   = np.hstack([X_val, coords_val])
X_test_spatial  = np.hstack([X_test, coords_test])

# ---------- Train XGBoost ----------
model = XGBRegressor(n_estimators=500, max_depth=3, learning_rate=0.1, random_state=42)
model.fit(X_train_spatial, y_train)

# ---------- Evaluate ----------
from sklearn.metrics import r2_score, mean_absolute_error
y_test_pred = model.predict(X_test_spatial)
print(f"Test R²:  {r2_score(y_test, y_test_pred):.4f}")
print(f"Test MAE: {mean_absolute_error(y_test, y_test_pred):.4f}")

---
## 3. GeoCP — Geographic Distance Only

GeoCP uses **geographic proximity** to weight calibration residuals.
Nearby calibration points contribute more to the uncertainty estimate of each test point.

**Key parameters:**
- `predict_f`: your model's `.predict` function
- `bandwidth`: controls the Gaussian kernel decay (how far geographic influence reaches)
- `miscoverage_level`: 0.1 means 90% prediction intervals

In [ ]:
geocp = GeoConformalSpatialRegression(
    predict_f=model.predict,
    miscoverage_level=0.1,         # 90% prediction interval
    bandwidth=5000,                # in coordinate units (meters for UTM)
    coord_calib=coords_calib,      # calibration coordinates (n, 2)
    coord_test=coords_test,        # test coordinates (m, 2)
    X_calib=X_calib_spatial,       # calibration features (for prediction)
    y_calib=y_calib,               # calibration true values
    X_test=X_test_spatial,         # test features (for prediction)
    y_test=y_test,                 # test true values
)

results_geocp = geocp.analyze()

print(f"Coverage probability: {results_geocp.coverage_probability:.3f}  (target: 0.90)")
print(f"Mean uncertainty:     {results_geocp.uncertainty:.4f}")
print(f"Mean interval width:  {np.mean(results_geocp.upper_bound - results_geocp.lower_bound):.4f}")

---
## 4. GeoSIMCP — Geographic Distance + Feature Similarity

GeoSIMCP extends GeoCP by also considering **feature-space similarity**.
This is especially useful when the spatial process is **nonstationary** — e.g., 
two nearby houses in different land-use zones may have very different price dynamics.

**Additional parameters over GeoCP:**
- `lambda_weight`: trade-off between geography (1.0) and features (0.0)
- `distance_metric`: `'euclidean'` or `'mnd'` (Minimum Normalized Difference)
- `standardize_weights`: whether to z-score normalize features for distance computation

### 4a. GeoSIMCP with Euclidean feature distance

In [ ]:
geosimcp_euc = GeoSIMConformalSpatialRegression(
    predict_f=model.predict,
    miscoverage_level=0.1,
    bandwidth=5000,
    coord_calib=coords_calib,
    coord_test=coords_test,
    X_calib=X_calib_spatial,       # features for prediction
    y_calib=y_calib,
    X_test=X_test_spatial,
    y_test=y_test,

    # --- GeoSIMCP-specific ---
    lambda_weight=0.5,             # 0.5 = equal weight to geo & feature distance
    distance_metric='euclidean',   # Euclidean feature distance
    standardize_weights=True,      # z-score normalize features
    X_calib_weight=X_calib,        # use only non-spatial features for distance
    X_test_weight=X_test,          # (exclude UTM coords from feature distance)
)

results_simcp_euc = geosimcp_euc.analyze()

print("GeoSIMCP-EUC (lambda=0.5):")
print(f"  Coverage:       {results_simcp_euc.coverage_probability:.3f}")
print(f"  Mean uncertainty: {results_simcp_euc.uncertainty:.4f}")
print(f"  Mean interval:  {np.mean(results_simcp_euc.upper_bound - results_simcp_euc.lower_bound):.4f}")

### 4b. GeoSIMCP with MND feature distance

MND (Minimum Normalized Difference) focuses on the **single most dissimilar feature dimension**.
It is more sensitive to local attribute outliers, making it robust when one dominant feature drives location differences.

In [ ]:
geosimcp_mnd = GeoSIMConformalSpatialRegression(
    predict_f=model.predict,
    miscoverage_level=0.1,
    bandwidth=5000,
    coord_calib=coords_calib,
    coord_test=coords_test,
    X_calib=X_calib_spatial,
    y_calib=y_calib,
    X_test=X_test_spatial,
    y_test=y_test,

    lambda_weight=0.5,
    distance_metric='mnd',         # Minimum Normalized Difference
    X_calib_weight=X_calib,
    X_test_weight=X_test,
)

results_simcp_mnd = geosimcp_mnd.analyze()

print("GeoSIMCP-MND (lambda=0.5):")
print(f"  Coverage:       {results_simcp_mnd.coverage_probability:.3f}")
print(f"  Mean uncertainty: {results_simcp_mnd.uncertainty:.4f}")
print(f"  Mean interval:  {np.mean(results_simcp_mnd.upper_bound - results_simcp_mnd.lower_bound):.4f}")

### Compare the three methods

In [ ]:
def interval_score(upper, lower, y_true, alpha=0.1):
    """Compute mean interval score (lower is better)."""
    width = upper - lower
    penalty = (2.0 / alpha) * (
        np.maximum(lower - y_true, 0) +
        np.maximum(y_true - upper, 0)
    )
    return np.mean(width + penalty)

rows = []
for name, r in [("GeoCP", results_geocp),
                ("GeoSIMCP-EUC", results_simcp_euc),
                ("GeoSIMCP-MND", results_simcp_mnd)]:
    rows.append({
        "Method": name,
        "Coverage": f"{r.coverage_probability:.3f}",
        "Mean Interval": f"{np.mean(r.upper_bound - r.lower_bound):.4f}",
        "Interval Score": f"{interval_score(r.upper_bound, r.lower_bound, y_test):.4f}",
    })

pd.DataFrame(rows)

---
## 5. Hyperparameter Tuning

The performance of GeoCP and GeoSIMCP depends heavily on hyperparameter choices.
We use **grid search** to find the optimal parameters that minimize the **interval score**
while maintaining valid coverage (>= 90%).

**Important**: Tuning is performed on the **validation set**, not the test set, to avoid data leakage.

**Interval Score** = interval width + penalty for miscoverage. Lower is better.

### 5a. Tune GeoCP (bandwidth only)

In [ ]:
bandwidths = np.linspace(500, 30000, 30)
geocp_results_list = []

for bw in bandwidths:
    geo = GeoConformalSpatialRegression(
        predict_f=model.predict, miscoverage_level=0.1, bandwidth=bw,
        coord_calib=coords_calib, coord_test=coords_val,
        X_calib=X_calib_spatial, y_calib=y_calib,
        X_test=X_val_spatial, y_test=y_val,
    )
    r = geo.analyze()
    score = interval_score(r.upper_bound, r.lower_bound, y_val)
    geocp_results_list.append({
        'bandwidth': bw, 'coverage': r.coverage_probability,
        'interval_score': score,
        'mean_interval': np.mean(r.upper_bound - r.lower_bound),
    })

df_geocp = pd.DataFrame(geocp_results_list)

# Select best: coverage >= 0.9, then lowest interval score
valid = df_geocp[df_geocp['coverage'] >= 0.9]
if len(valid) > 0:
    best_geocp = valid.loc[valid['interval_score'].idxmin()]
else:
    best_geocp = df_geocp.loc[df_geocp['interval_score'].idxmin()]

print("=== Best GeoCP (tuned on validation set) ===")
print(f"  Bandwidth:      {best_geocp['bandwidth']:.0f}")
print(f"  Coverage:       {best_geocp['coverage']:.3f}")
print(f"  Interval Score: {best_geocp['interval_score']:.4f}")
print(f"  Mean Interval:  {best_geocp['mean_interval']:.4f}")

In [ ]:
# ---------- Visualize bandwidth tuning ----------
fig, ax1 = plt.subplots(figsize=(10, 4))

ax1.plot(df_geocp['bandwidth'], df_geocp['interval_score'], 'b-o', markersize=4, label='Interval Score')
ax1.set_xlabel('Bandwidth (meters)')
ax1.set_ylabel('Interval Score', color='b')
ax1.axvline(best_geocp['bandwidth'], color='red', linestyle='--', alpha=0.7, label=f"Best bw={best_geocp['bandwidth']:.0f}")

ax2 = ax1.twinx()
ax2.plot(df_geocp['bandwidth'], df_geocp['coverage'], 'g-s', markersize=4, label='Coverage')
ax2.axhline(0.9, color='gray', linestyle=':', alpha=0.5)
ax2.set_ylabel('Coverage', color='g')

ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.title('GeoCP: Bandwidth Tuning')
plt.tight_layout()
plt.show()

### 5b. Tune GeoSIMCP (bandwidth + lambda)

GeoSIMCP has two hyperparameters to tune:
- **bandwidth**: same as GeoCP
- **lambda_weight**: trade-off between geography (1) and features (0)

In [ ]:
bandwidths_sim = np.linspace(500, 30000, 15)
lambdas = np.arange(0, 1.05, 0.1)
simcp_results_list = []

total = len(bandwidths_sim) * len(lambdas)
print(f"Grid search: {len(bandwidths_sim)} bandwidths x {len(lambdas)} lambdas = {total} configs")

for bw in bandwidths_sim:
    for lam in lambdas:
        geo = GeoSIMConformalSpatialRegression(
            predict_f=model.predict, miscoverage_level=0.1,
            bandwidth=bw, lambda_weight=lam,
            distance_metric='euclidean', standardize_weights=True,
            coord_calib=coords_calib, coord_test=coords_val,
            X_calib=X_calib_spatial, y_calib=y_calib,
            X_test=X_val_spatial, y_test=y_val,
            X_calib_weight=X_calib, X_test_weight=X_val,
        )
        r = geo.analyze()
        score = interval_score(r.upper_bound, r.lower_bound, y_val)
        simcp_results_list.append({
            'bandwidth': bw, 'lambda': lam,
            'coverage': r.coverage_probability,
            'interval_score': score,
            'mean_interval': np.mean(r.upper_bound - r.lower_bound),
        })

df_simcp = pd.DataFrame(simcp_results_list)

# Select best: coverage >= 0.9, then lowest interval score
valid_sim = df_simcp[df_simcp['coverage'] >= 0.9]
if len(valid_sim) > 0:
    best_simcp = valid_sim.loc[valid_sim['interval_score'].idxmin()]
else:
    best_simcp = df_simcp.loc[df_simcp['interval_score'].idxmin()]

print(f"\n=== Best GeoSIMCP-EUC (tuned on validation set) ===")
print(f"  Bandwidth:      {best_simcp['bandwidth']:.0f}")
print(f"  Lambda:         {best_simcp['lambda']:.2f}")
print(f"  Coverage:       {best_simcp['coverage']:.3f}")
print(f"  Interval Score: {best_simcp['interval_score']:.4f}")
print(f"  Mean Interval:  {best_simcp['mean_interval']:.4f}")

In [ ]:
# ---------- Visualize grid search as heatmap ----------
pivot = df_simcp.pivot_table(index='lambda', columns='bandwidth', values='interval_score')

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(pivot.values, aspect='auto', cmap='viridis_r',
               extent=[pivot.columns.min(), pivot.columns.max(),
                       pivot.index.max(), pivot.index.min()])
ax.set_xlabel('Bandwidth (meters)')
ax.set_ylabel('Lambda (1=geo, 0=feature)')
ax.set_title('GeoSIMCP-EUC: Interval Score (lower = better)')

# Mark best point
ax.plot(best_simcp['bandwidth'], best_simcp['lambda'], 'r*', markersize=20,
        label=f"Best: bw={best_simcp['bandwidth']:.0f}, λ={best_simcp['lambda']:.2f}")
ax.legend(fontsize=11)

plt.colorbar(im, ax=ax, label='Interval Score')
plt.tight_layout()
plt.show()

---
## 6. Run Best Models & Compare

Re-run GeoCP and GeoSIMCP with their optimal hyperparameters (tuned on validation set), and evaluate on the **test set**.

In [ ]:
# ---------- Best GeoCP ----------
geocp_best = GeoConformalSpatialRegression(
    predict_f=model.predict, miscoverage_level=0.1,
    bandwidth=best_geocp['bandwidth'],
    coord_calib=coords_calib, coord_test=coords_test,
    X_calib=X_calib_spatial, y_calib=y_calib,
    X_test=X_test_spatial, y_test=y_test,
)
results_best_geocp = geocp_best.analyze()

# ---------- Best GeoSIMCP ----------
geosimcp_best = GeoSIMConformalSpatialRegression(
    predict_f=model.predict, miscoverage_level=0.1,
    bandwidth=best_simcp['bandwidth'],
    lambda_weight=best_simcp['lambda'],
    distance_metric='euclidean', standardize_weights=True,
    coord_calib=coords_calib, coord_test=coords_test,
    X_calib=X_calib_spatial, y_calib=y_calib,
    X_test=X_test_spatial, y_test=y_test,
    X_calib_weight=X_calib, X_test_weight=X_test,
)
results_best_simcp = geosimcp_best.analyze()

# ---------- Summary table ----------
summary = []
for name, r in [("GeoCP (tuned)", results_best_geocp),
                ("GeoSIMCP-EUC (tuned)", results_best_simcp)]:
    summary.append({
        "Method": name,
        "Coverage": f"{r.coverage_probability:.3f}",
        "Mean Interval": f"{np.mean(r.upper_bound - r.lower_bound):.4f}",
        "Interval Score": f"{interval_score(r.upper_bound, r.lower_bound, y_test):.4f}",
    })
pd.DataFrame(summary)

---
## 7. Spatial Mapping

Visualize the prediction results and uncertainty on a map.
The `to_gpd()` method converts results to a GeoDataFrame for easy plotting.

In [ ]:
def build_gdf(results, coords, crs='EPSG:32610'):
    """Build a GeoDataFrame from results and UTM coordinates."""
    gdf = gpd.GeoDataFrame(
        {
            'pred': results.pred,
            'uncertainty': results.geo_uncertainty,
            'upper': results.upper_bound,
            'lower': results.lower_bound,
        },
        geometry=gpd.points_from_xy(coords[:, 0], coords[:, 1]),
        crs=crs,
    )
    # Reproject to Web Mercator for contextily basemap
    return gdf.to_crs(epsg=3857)

gdf_geocp  = build_gdf(results_best_geocp, coords_test)
gdf_simcp  = build_gdf(results_best_simcp, coords_test)

### 7a. Predicted Values vs. Uncertainty (GeoCP)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Predicted values ---
gdf_geocp.plot(column='pred', cmap='RdYlBu_r', legend=True, ax=axes[0],
               markersize=20, alpha=0.8,
               legend_kwds={'label': 'Predicted log(price)', 'shrink': 0.6})
cx.add_basemap(axes[0], source=cx.providers.CartoDB.Positron, alpha=0.5)
axes[0].set_title('GeoCP: Predicted Values', fontsize=13)
axes[0].set_axis_off()

# --- Uncertainty ---
gdf_geocp.plot(column='uncertainty', cmap='Reds', legend=True, ax=axes[1],
               markersize=20, alpha=0.8,
               legend_kwds={'label': 'Uncertainty', 'shrink': 0.6})
cx.add_basemap(axes[1], source=cx.providers.CartoDB.Positron, alpha=0.5)
axes[1].set_title('GeoCP: Prediction Uncertainty', fontsize=13)
axes[1].set_axis_off()

plt.suptitle(f"GeoCP (bandwidth={best_geocp['bandwidth']:.0f})", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### 7b. Predicted Values vs. Uncertainty (GeoSIMCP)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Predicted values ---
gdf_simcp.plot(column='pred', cmap='RdYlBu_r', legend=True, ax=axes[0],
               markersize=20, alpha=0.8,
               legend_kwds={'label': 'Predicted log(price)', 'shrink': 0.6})
cx.add_basemap(axes[0], source=cx.providers.CartoDB.Positron, alpha=0.5)
axes[0].set_title('GeoSIMCP: Predicted Values', fontsize=13)
axes[0].set_axis_off()

# --- Uncertainty ---
gdf_simcp.plot(column='uncertainty', cmap='Reds', legend=True, ax=axes[1],
               markersize=20, alpha=0.8,
               legend_kwds={'label': 'Uncertainty', 'shrink': 0.6})
cx.add_basemap(axes[1], source=cx.providers.CartoDB.Positron, alpha=0.5)
axes[1].set_title('GeoSIMCP: Prediction Uncertainty', fontsize=13)
axes[1].set_axis_off()

plt.suptitle(f"GeoSIMCP-EUC (bandwidth={best_simcp['bandwidth']:.0f}, λ={best_simcp['lambda']:.2f})",
             fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### 7c. Side-by-side Uncertainty Comparison: GeoCP vs GeoSIMCP

In [ ]:
# Use shared colorbar limits for fair comparison
vmin = min(gdf_geocp['uncertainty'].min(), gdf_simcp['uncertainty'].min())
vmax = max(gdf_geocp['uncertainty'].max(), gdf_simcp['uncertainty'].max())

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

gdf_geocp.plot(column='uncertainty', cmap='Reds', legend=True, ax=axes[0],
               markersize=20, alpha=0.8, vmin=vmin, vmax=vmax,
               legend_kwds={'label': 'Uncertainty', 'shrink': 0.6})
cx.add_basemap(axes[0], source=cx.providers.CartoDB.Positron, alpha=0.5)
axes[0].set_title(f"GeoCP (bw={best_geocp['bandwidth']:.0f})", fontsize=13)
axes[0].set_axis_off()

gdf_simcp.plot(column='uncertainty', cmap='Reds', legend=True, ax=axes[1],
               markersize=20, alpha=0.8, vmin=vmin, vmax=vmax,
               legend_kwds={'label': 'Uncertainty', 'shrink': 0.6})
cx.add_basemap(axes[1], source=cx.providers.CartoDB.Positron, alpha=0.5)
axes[1].set_title(f"GeoSIMCP-EUC (bw={best_simcp['bandwidth']:.0f}, λ={best_simcp['lambda']:.2f})", fontsize=13)
axes[1].set_axis_off()

plt.suptitle('Uncertainty Comparison: GeoCP vs GeoSIMCP', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### 7d. Prediction Intervals for Individual Test Points

In [ ]:
# Show prediction intervals for 30 random test points
n_show = 30
idx = np.random.RandomState(42).choice(len(y_test), n_show, replace=False)
idx = idx[np.argsort(y_test[idx])]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for ax, r, title in [(axes[0], results_best_geocp, 'GeoCP'),
                      (axes[1], results_best_simcp, 'GeoSIMCP-EUC')]:
    x_pos = np.arange(n_show)
    ax.fill_between(x_pos, r.lower_bound[idx], r.upper_bound[idx],
                    alpha=0.3, color='steelblue', label='90% interval')
    ax.plot(x_pos, r.pred[idx], 'b-o', markersize=4, label='Predicted')
    ax.plot(x_pos, y_test[idx], 'r-x', markersize=6, label='Actual')
    ax.set_xlabel('Test sample (sorted by true value)')
    ax.set_ylabel('log(price)')
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=9)

plt.suptitle('Prediction Intervals for 30 Test Points', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Summary

| Step | GeoCP | GeoSIMCP |
|------|-------|----------|
| **Class** | `GeoConformalSpatialRegression` | `GeoSIMConformalSpatialRegression` |
| **Distance** | Geographic only | Geographic + Feature similarity |
| **Key params** | `bandwidth` | `bandwidth` + `lambda_weight` + `distance_metric` |
| **Tune** | Grid search over `bandwidth` | Grid search over `bandwidth` x `lambda_weight` |
| **Best for** | Spatial interpolation, stationary processes | Nonstationary processes, feature-rich data |
| **Special case** | — | `lambda_weight=1.0` reduces to GeoCP |